# Stage 2 - metamorphic expansion

18 follow-ups derived from 8 of the 11 seeds: 2 each, and 3 from
`scheduling_move_tram` and `venue_lookup_seafood_lisbon`. One relation each,
from the catalogue its function permits (D22). Budget per D21, extended by
D50/D51/D61 where a case was the only route to a tool or a relation.

Three seeds take no follow-up. The two forecast seeds and the surf seed exist
to reach tools nothing else reaches (D51, D54); perturbing them adds no
coverage.

A follow-up is a seed with a perturbed request and an expected delta stated in
the seed's own constraint vocabulary. Perturbation and delta are hand-chosen
(`cases/derivations.py`); the derived case's shape is `src/metamorphic.py` -
inheritance, implied predicates, `n = 4` against a seed's `n = 8`, base id.

Suite: **29 cases, 160 executions** at full `n`.

## What the relations can and cannot claim

**`search_places` caps at 5 results per call.** "Narrow the request, expect
fewer results" is therefore not a relation this system has: a tighter
constraint makes the agent search again and `places_surfaced` rises. Narrowing
changes *which* venues return; asking for a second kind of place changes the
answer's shape. The suite asserts those two, not the count.

### Half the suite has no structural delta, by construction

A structural check sees only what the agent **did** — which tools it called and
what it wrote to the trip. It never reads the reply.

For venue lookup and scheduling, a perturbation changes those things. "Move the
tram ride to Sunday instead of Saturday" writes a different day into the
itinerary, and `slot_moved` fails an agent that writes the wrong one.

For factual QA and advisory it changes neither. Compare the Alfama seed with
its DIR-1 follow-up:

| | seed | follow-up |
|---|---|---|
| asks | *what's the history of Alfama, briefly?* | *what happened to Alfama in the 1755 earthquake?* |
| tool called | `read_wikipedia` | `read_wikipedia` |
| written to the trip | nothing | nothing |
| structural verdict | `none` | `none` |

To the checker the two executions are indistinguishable. The follow-up inherits
the seed's constraints and they hold either way — **an agent that ignored the
earthquake entirely and repeated its answer about Alfama's history would pass
every structural check in the case.** Only the judge can separate them, because
only the judge reads the reply.

This is not a defect in those cases; it is what D22's eligibility means in
practice. It affects **7 of the 11 seeds and 8 of the 18 follow-ups** — factual
QA may take only DIR-1 besides the invariance relations, and advisory only
invariance, so neither function has a relation that changes observable
behaviour.

Two consequences worth stating before any results are read:

- For those cases a `none` verdict means *nothing broke*, **not** *the
  perturbation was answered*. A structural pass rate over the whole suite
  therefore overstates what was verified, and should be reported split by
  whether the case has a structural delta at all.
- Their entire evidential weight sits on the judge, which is why the judge's
  reference material and blindness rules (P3, D34, D35) matter more for this
  half than for the other.

This is the boundary between structural verification and scoring.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parents[1]))  # the subtree, so `src` imports

import json
from collections import Counter

import yaml

from src.cases import CASES_DIR, load_cases, suite_version
from src.metamorphic import ELIGIBLE, RELATION_NAMES, derive
from cases.derivations import DERIVATIONS

## Deriving the eighteen

Regenerated from the seeds every time this runs, so a seed edit cannot leave a stale follow-up behind.

In [2]:
seeds = {c.case_id: c for c in load_cases() if c.provenance == "seeded"}
# Two follow-ups per seed (D21), with two exceptions the suite records
# (D50, D51): `scheduling_move_tram` takes three, because `schedule_block`
# and `unschedule_block` are reachable only under /reschedule and so both
# cases that reach them must derive from that seed; and
# `factual_qa_lisbon_surf_conditions` takes none, because it exists to
# reach `get_climate_normals` and a perturbation of it would buy no tool
# the suite does not already have.
assert set(DERIVATIONS) <= set(seeds)

written = []
for seed_id, pair in DERIVATIONS.items():
    for d in pair:
        data = derive(seeds[seed_id], d)
        path = CASES_DIR / f"{data['case_id']}.yaml"
        path.write_text(
            yaml.dump(data, sort_keys=False, allow_unicode=True,
                      default_flow_style=False, width=100),
            encoding="utf-8",
        )
        written.append(data["case_id"])

cases = load_cases()
expanded = [c for c in cases if c.provenance == "metamorphic"]
print(f"{len(written)} follow-ups derived; {len(cases)} cases in the suite")
print(f"executions per run: {sum(c.repetitions_n for c in cases)}  (D21 budget: 128)")
print(f"relations: {dict(sorted(Counter(c.metamorphic_relation for c in expanded).items()))}")
print(f"suite_version: {suite_version()[:16]}…")

18 follow-ups derived; 29 cases in the suite
executions per run: 160  (D21 budget: 128)
relations: {'DIR-1': 5, 'DIR-2': 3, 'DIR-3': 1, 'DIR-4': 2, 'INV-1': 4, 'INV-2': 1, 'INV-3': 1, 'INV-4': 1}
suite_version: 031589fd96588100…


## Eligibility

Relation counts are uneven (DIR-1 five, DIR-3 one, INV-2 one). The denominator
accounts for most of it: D22 permits a relation per function, so opportunity is
uneven before any choice is made.

**Eligible** - DIR-2 on `scheduling_move_tram`. The seed asks for Saturday
afternoon, the follow-up for Sunday, and `slot_scheduled` moves with it. The
expected outcome is a different day: one specific, checkable failure.

**Not eligible** - DIR-2 on `factual_qa_alfama_history`. "What happened in the
1755 earthquake" has no day to change and no answer a changed day would alter;
the expected delta would be undefined.

DIR-2/3/4 are eligible on 4 of 11 seeds, INV-1..4 on all 11.

Within eligibility, directional relations were preferred: 25-75% of opportunity
against 9-36% for invariance. A directional relation asserts the answer
*changes* and fails specifically; invariance asserts sameness, which language
models are robust to, so an invariance-weighted suite reports its own
insensitivity as coverage.

INV-2/3/4 carry one case each - a floor to instantiate the catalogue, not a
sample. No per-relation statistic follows from them.

In [3]:
from src import tables

# The grid, not a list per function: the question a reader has is "why does
# THIS relation have so few cases", and that is a column.
tables.eligibility()

,perturbation,advisory,factual_qa,scheduling,venue_lookup,eligible seeds
relation,,,,,,
DIR-1,constraint added or changed,-,yes,yes,yes,9
DIR-2,day changed,-,-,yes,yes,4
DIR-3,count changed,-,-,yes,yes,4
DIR-4,negation,-,-,yes,yes,4
INV-1,paraphrase,yes,yes,yes,yes,11
INV-2,typos and casing,yes,yes,yes,yes,11
INV-3,irrelevant detail added,yes,yes,yes,yes,11
INV-4,language switch (German),yes,yes,yes,yes,11


In [4]:
# Use against opportunity, not against the other relations (D61).
# DIR-2/3/4 look starved only until you read their denominator.
tables.relations()

,perturbation,kind,eligible seeds,follow-ups,share of opportunity
relation,,,,,
DIR-1,constraint added or changed,directional,9,5,0.556
DIR-2,day changed,directional,4,3,0.750
DIR-3,count changed,directional,4,1,0.250
DIR-4,negation,directional,4,2,0.500
INV-1,paraphrase,invariance,11,4,0.364
INV-2,typos and casing,invariance,11,1,0.091
INV-3,irrelevant detail added,invariance,11,1,0.091
INV-4,language switch (German),invariance,11,1,0.091


In [5]:
# The limitation travels with the numbers rather than living only in prose.
from src.metamorphic import relation_report

print(relation_report().split("\n\n", 1)[-1])

single instance: INV-2, INV-3, INV-4 — instantiated and exercised, but one case is not a sample. Nothing statistical may be claimed per-relation for these (D61).


## The tool-coverage metric

D21 budgeted 8 seeds and 2 follow-ups each; the suite is 11 and 18. Every case
beyond the budget was admitted by one metric.

Per tool: `cases requiring` (assert it ran), `cases permitting` (allow it at
`min: 0`), `calls`/`errors` (what a run did). The gap between the first two is
the measurement - a tool permitted by 11 cases and required by none is not
covered, because no case fails if it disappears. `tools_any_of` is read as the
disjunction it is (D66).

**Admission rule.** A case beyond the budget earns its place only by being the
only route to a tool or a relation, alone or jointly with a named few (D69).
Otherwise it is a second sample of a measurement the suite already has, and
raising `n` is cheaper. The table below is that rule, computed.

In [6]:
# The admission rule, made checkable: what would be lost if each case went.
# 'jointly with N' means N other cases also require it — delete them all and
# the tool is uncovered, which is still the only route to it (D69).
u = tables.unique_contribution()
u[u['carries coverage']][['provenance', 'route to', 'sole coverage']]

,provenance,route to,sole coverage
case,,,
advisory_packing_list__inv3_irrelevant_detail,metamorphic,relation INV-3,True
advisory_sintra_full_or_half_day__inv4_german,metamorphic,relation INV-4,True
factual_qa_lisbon_forecast_next_days,seeded,get_weather (jointly with 1),False
factual_qa_lisbon_surf_conditions,seeded,get_climate_normals,True
factual_qa_lisbon_trip_forecast,seeded,get_weather (jointly with 1),False
scheduling_belem_morning__dir3_a_full_day,metamorphic,relation DIR-3,True
scheduling_move_tram,seeded,move_slot (jointly with 1),False
scheduling_move_tram__dir1_put_the_monastery_on,metamorphic,schedule_block,True
scheduling_move_tram__dir2_sunday_afternoon,metamorphic,move_slot (jointly with 1),False


## The three seeds added after stage 1

`sim/01_cases` introduced eight seeds. Three more were added here, under the
coverage rule above rather than under D21's budget, and they have not been
shown anywhere until now — the prose referred to them while the notebook never
presented them.

Each was admitted for a tool no other case could reach:

- **`factual_qa_lisbon_surf_conditions`** (D51) — the only case that requires
  `get_climate_normals`. Before it, six of twelve tools were never called by
  any run.
- **`factual_qa_lisbon_forecast_next_days`** and
  **`factual_qa_lisbon_trip_forecast`** (D51, D54) — jointly the only cases
  requiring `get_weather`. They exist as a **pair** because the thing under
  test is which calendar the agent resolves against: one asks about the next
  few days and must be answered from *today*, the other asks about the trip and
  must be answered from the *trip's dates*. Neither can test that alone, since
  a case that does not say which period it means cannot mark either answer
  wrong. `trip_forecast` is the suite's only case on a near-dated fixture
  (D54), because a forecast exists only within sixteen days of being asked for.

Finding that `get_weather` had never once succeeded — 118 executions, every
call returning `Open-Meteo error: 400` — is what these seeds were written to
make visible, and what led to D56/D57 fixing the agent.

In [7]:
import json, textwrap

from src.cases import load_cases
from src.coverage import tools_required_by

ADDED = [
    "factual_qa_lisbon_surf_conditions",
    "factual_qa_lisbon_forecast_next_days",
    "factual_qa_lisbon_trip_forecast",
]
by_id = {c.case_id: c for c in load_cases()}

for case_id in ADDED:
    c = by_id[case_id]
    print("=" * 100)
    print(f"{c.case_id}   [{c.function} \u00b7 {c.provenance} \u00b7 priority {c.run_priority} \u00b7 n={c.repetitions_n}]")
    print(f"fixture:  {c.fixture_dates}" + ("   (dated relative to the run - D54)" if c.fixture_dates == "near" else ""))
    print("persona:  " + textwrap.fill(c.user_profile.strip(), 90, subsequent_indent=" " * 10))
    print("goal:     " + textwrap.fill(c.goal.strip(), 90, subsequent_indent=" " * 10))
    print(f"opens:    {c.opening_turn!r}" + (f"   (intent {c.intent})" if c.intent else ""))
    if c.expected_tool_sequence:
        expects = " -> ".join(c.expected_tool_sequence)
    elif c.declared_constraints.checks.tools_any_of:
        expects = "any one of: " + ", ".join(c.declared_constraints.checks.tools_any_of)
    else:
        expects = "nothing - which tools inform the answer is the agent's call (D29)"
    print("expects:  " + expects)
    print("requires: " + (", ".join(sorted(tools_required_by(c))) or "no tool unconditionally"))
    allowed = c.allowed_tools()
    print("permits:  " + (", ".join(sorted(allowed)) + "  (every other tool is a failure)"
                          if allowed else "any tool - the case constrains none"))
    cons = c.declared_constraints
    for k, v in cons.checks.model_dump(exclude_none=True).items():
        print(f"check:    {k} = {json.dumps(v)}")
    if cons.predicates:
        print("predicates: " + ", ".join(cons.predicates))
    for i, crit in enumerate(c.scoring_criteria, 1):
        print(f"criterion {i}: {crit}")
    for ref in c.references:
        print(f"reference: {ref.section}")
        print(f"           references/{ref.path} ({len(ref.text())} chars)")
    print("fixture_depends_on: " + (", ".join(c.fixture_depends_on)
                                    if c.fixture_depends_on else "nothing - the case reads nothing off the trip"))
    if c.notes:
        print("notes:    " + textwrap.fill(c.notes.strip(), 90, subsequent_indent=" " * 10))
    print()

factual_qa_lisbon_surf_conditions   [factual_qa · seeded · priority 1 · n=8]
fixture:  fixture
persona:  Jonas, 31, travelling with the group and surfs when he can. Asks a practical question and
          wants numbers, not encouragement.
goal:     Know what the waves and wind are like around Lisbon in November, to judge whether it is
          worth bringing a board. He wants the figures and nothing else; if Tonic offers
          to change anything on the trip he says no thanks.
opens:    '@Tonic what are the waves and wind like around Lisbon in November? thinking of bringing a board'
expects:  any one of: get_climate_normals
requires: get_climate_normals
permits:  get_climate_normals, get_itinerary, get_region_climate, get_trip_details, get_weather, list_trip_blocks  (every other tool is a failure)
check:    tool_counts = {"get_region_climate": {"min": 0, "max": 1}, "get_weather": {"min": 0, "max": 1}, "get_climate_normals": {"min": 1, "max": 2}}
check:    tools_any_of = ["get_clima

## Each follow-up beside its seed

The request, what changed, and the delta the case claims.

In [8]:
def show(seed, follow):
    checks_seed = seed.declared_constraints.checks.model_dump(exclude_none=True)
    checks_follow = follow.declared_constraints.checks.model_dump(exclude_none=True)
    changed = {k: v for k, v in checks_follow.items() if checks_seed.get(k) != v}
    dropped = [k for k in checks_seed if k not in checks_follow]

    print("=" * 100)
    print(f"{follow.case_id}")
    print(f"  {follow.metamorphic_relation} — {RELATION_NAMES[follow.metamorphic_relation]}"
          f"   (base: {seed.case_id}, n={follow.repetitions_n})")
    print(f"  seed asked   : {seed.opening_turn}")
    print(f"  now asks     : {follow.opening_turn}")
    if changed:
        print("  delta claimed:")
        for k, v in changed.items():
            print(f"      {k} = {json.dumps(v, ensure_ascii=False)}")
    else:
        print("  delta claimed: none structurally — the change is in the answer, and the judge scores it")
    if dropped:
        print(f"  dropped      : {dropped}  (the perturbation makes these untrue)")
    if follow.declared_constraints.predicates:
        print(f"  predicates   : {follow.declared_constraints.predicates}")
    print(f"  why          : {follow.notes}")


by_id = {c.case_id: c for c in cases}
for seed_id, pair in DERIVATIONS.items():
    for d in pair:
        show(seeds[seed_id], by_id[f"{seed_id}__{d.suffix}"])

venue_lookup_seafood_lisbon__dir1_and_a_hotel
  DIR-1 — constraint added or changed   (base: venue_lookup_seafood_lisbon, n=4)
  seed asked   : /find good seafood places near Cais do Sodré, nothing too pricey
  now asks     : /find good seafood places near Cais do Sodré, nothing too pricey — and somewhere to stay nearby too
  delta claimed:
      places_surfaced = {"min": 2}
      places_named = {"min": 3}
      places_named_types = ["restaurant", "lodging"]
  why          : The strongest perturbation available to a venue-lookup case: asking for a second kind of place changes the shape of the answer rather than its wording. Chosen over the obvious 'ask for something narrower and expect fewer results', which is not a relation this system has — search_places is capped at five per call, so narrowing pushes the agent to search again and the count rises.
venue_lookup_seafood_lisbon__dir4_not_a_meal
  DIR-4 — negation   (base: venue_lookup_seafood_lisbon, n=4)
  seed asked   : /find good sea

## What a run of all twenty-nine produced

One repetition each, so every follow-up has its base execution at the same
repetition to compare against (which is what the predicates need).

Not a baseline: `n = 1`, no judge, and the run row says so.

In [9]:
from src.checker import check_run
from src import execution as exmod
from src.db import admin

# Pinned, not "the newest run". The outputs stored in this notebook describe
# ONE run, and a selector that drifts would leave the prose describing numbers
# that are no longer below it. Re-point it deliberately when re-running.
RUN_ID = "09f288e9-b39b-4308-a66f-d769b2c05d56"

run_id = RUN_ID
meta = admin().table("study_runs").select(
    "notes, started_at, suite_version, api_version, api_commit, git_sha"
).eq("id", run_id).execute().data[0]
print(f"run {run_id[:8]}  {meta['started_at'][:19]}")
print(f"  {meta['notes']}")
print(f"  suite {meta['suite_version'][:12]}  api {meta['api_version']} @ {meta['api_commit']}")

run 09f288e9  2026-09-22T19:26:23
  FULL SUITE n=1 post-D59..D65 - retry policy, INV-2, versions, tables
  suite 1419b4d9de54  api 0.1.1 @ 14f37060eb1f


In [10]:
results = check_run(run_id, persist=True)

clean = sum(1 for _, v in results if v.outcome == "none")
print(f"{clean}/{len(results)} executions with no structural finding")

# Sorted so anything with a finding is at the top: a clean run is a long list
# of rows nobody needs to read, and the exceptions are the whole point.
tables.structural(results).sort_values(["findings", "kind"], ascending=[False, True])

29/29 executions with no structural finding


,,,kind,outcome,findings,detail
case,rep,attempt,,,,
factual_qa_alfama_history__dir1_the_earthquake,1,1,DIR-1,none,0,
factual_qa_lisbon_november_weather__dir1_july_instead,1,1,DIR-1,none,0,
scheduling_move_tram__dir1_put_the_monastery_on,1,1,DIR-1,none,0,
venue_lookup_seafood_lisbon__dir1_and_a_hotel,1,1,DIR-1,none,0,
venue_lookup_sintra_lunch__dir1_coffee_not_lunch,1,1,DIR-1,none,0,
scheduling_belem_morning__dir2_on_the_9th,1,1,DIR-2,none,0,
scheduling_move_tram__dir2_sunday_afternoon,1,1,DIR-2,none,0,
venue_lookup_sintra_lunch__dir2_lisbon_day,1,1,DIR-2,none,0,
scheduling_belem_morning__dir3_a_full_day,1,1,DIR-3,none,0,


## Tool coverage for this run

Observed against declared. `all errored` is the column that
found the weather defect: a tool can be declared, called, and broken,
while every case stays green.

In [11]:
tables.tool_coverage(run_id)

,cases requiring,cases permitting,calls,errors,all errored,parameters exercised
tool,,,,,,
get_climate_normals,1,6,1,0,False,"include_marine, lat, lng"
get_region_climate,0,6,6,0,False,place
get_weather,2,13,3,1,False,"end_date, lat, lng, start_date"
move_slot,2,2,2,0,False,"day_date, end_time, slot_id, start_time"
propose_plan,3,3,3,0,False,"days, description, title"
read_wikipedia,3,3,6,0,False,"query, section"
schedule_block,1,1,1,0,False,"block_id, day_date, end_time, start_time"
search_places,10,10,23,0,False,"lat, lng, query"
unschedule_block,1,1,1,0,False,slot_id


## Findings

Two kinds, kept apart because they are not the same evidence. Defects in the
system under test are what the pipeline exists to produce. Defects in the
harness are instrument maturation and support no claim about the agent.

**System under test**

| finding | surfaced by | ref |
|---|---|---|
| `get_weather` never succeeded in 118 executions | coverage: declared, called, every call errored | D56 |
| the agent was never told the current date | a case *requiring* `get_weather`'s success path | D57 |
| `get_region_climate` and `get_weather` instructions contradicted each other | the same case answered with averages for a trip 3 days out | D57 |
| an out-of-horizon call returned a bare `400`, and the agent retried it | tool error text in the trace | D56 |
| 6 of 12 tools never called | coverage, first pass | D50, D51 |
| an empty reply conflated 3 distinct failures | the failure report's evidence columns | D59 |
| the capacity retry could consume the whole turn | latency split by `retry_count` | D60 |
| a prompt change altered venue-search behaviour | a structural check failing across runs | D72 |

Effect of the weather fixes on the two cases that provoked them: 1/10 to 10/10
and 3/10 to 9/10, with `get_weather` making one call each and zero errors.

**Harness.** The reset destroyed the record it was meant to preserve (D38,
L08); the checker served one attempt's verdict to another (D65); coverage
over-counted disjunctive requirements (D66); scoring counted a superseded
attempt (D59); resume collided on the attempt key (D71). Listed separately -
folding them into the table above would overstate the §3.1 claim.

## Did the perturbations actually perturb?

A directional relation claims the answer changes. The check above says whether
each execution matched its own constraints; this says whether the follow-up
differed from its seed at all — which is the part a passing verdict can hide.

A follow-up whose venue set is identical to its seed's has not been perturbed,
whatever its checks say.

In [12]:
# `overlap` is the column to read, and it means opposite things by kind:
# for a DIRECTIONAL relation a high overlap is a warning (the request changed
# and the answer did not), for an INVARIANCE relation it is the expected
# result. `suspect` flags only the first of those.
tables.perturbation(results)

,relation,kind,places,seed places,shared,overlap,tools,suspect
follow-up,,,,,,,,
advisory_packing_list__inv1_paraphrase,INV-1,invariance,0,0,0,NaN,same,False
advisory_packing_list__inv3_irrelevant_detail,INV-3,invariance,0,0,0,NaN,differ,False
advisory_sintra_full_or_half_day__inv1_paraphrase,INV-1,invariance,0,0,0,NaN,same,False
advisory_sintra_full_or_half_day__inv4_german,INV-4,invariance,0,0,0,NaN,same,False
factual_qa_alfama_history__dir1_the_earthquake,DIR-1,directional,0,0,0,NaN,same,False
factual_qa_alfama_history__inv1_paraphrase,INV-1,invariance,0,0,0,NaN,same,False
factual_qa_lisbon_november_weather__dir1_july_instead,DIR-1,directional,0,0,0,NaN,same,False
factual_qa_lisbon_november_weather__inv1_paraphrase,INV-1,invariance,0,0,0,NaN,same,False
scheduling_belem_morning__dir2_on_the_9th,DIR-2,directional,5,2,2,0.40,same,False


## Review

What to check in the cases above:

1. **Is the perturbed request what a member would type?** The wording is the
   whole input; a stilted one tests a sentence nobody writes.
2. **Is the claimed delta the right claim?** Asserting too much manufactures
   false regressions (D30); too little passes whatever the agent does.
3. **The light cases** - factual QA's and advisory's paraphrases - assert
   nothing structurally. Decide whether they earn their executions.
4. **`INV-2`** is the weakest relation in the suite and deliberately the only
   one of its kind (D61). It sits on the venue lookup, the closest the suite
   gets to a typo reaching a tool. Measured: the agent authors the
   `search_places` query and normalised every typo, searching a string
   byte-identical to the seed's; the case scored 10/10. The relation is weak
   for a specific reason - query rewriting, not tolerant matching downstream -
   which narrows where to look if a typo case ever fails.

Edits go in `cases/derivations.py`; this notebook regenerates the YAML, which
is never hand-edited.